In [ ]:
project_path = "/home/jupyter"
import os
import sys

sys.path.append(project_path)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from google.cloud import bigquery

from fintrans_toolbox.src import bq_utils as bq
from fintrans_toolbox.src import table_utils as t


client = bigquery.Client()


In [ ]:
# Summarise the data by UK Cardholder Domestic Spending All Quarterly

UK_spending_by_Dom_All = '''SELECT time_period_value, destination_country, spend 
FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` 
where time_period = 'Quarter' 
and mcc = 'All' 
and mcg = 'All' 
and merchant_channel = 'All' 
and cardholder_origin_country = 'All' 
and cardholder_origin = 'UNITED KINGDOM' 
and destination_country = 'UNITED KINGDOM' 
GROUP BY destination_country, 
time_period_value, spend 
ORDER BY time_period_value, destination_country DESC'''
df_by_Dom_All = bq.read_bq_table_sql(client, UK_spending_by_Dom_All)
df_by_Dom_All.head()

# Caculate UK Domestic Total Spending Quarterly

# Assuming df_by_Dom_All is the DataFrame returned from the BigQuery query
# Then group by 'time_period_value' and sum the 'spend' for each quarter

# Check if df_by_Dom_All is not None and has the expected columns
if df_by_Dom_All is not None and 'time_period_value' in df_by_Dom_All.columns and 'spend' in df_by_Dom_All.columns:
    # Group by quarter and sum the spend
    UK_spending_by_Dom_All = df_by_Dom_All.groupby('time_period_value')['spend'].sum().reset_index()
   
 # Rename the column
    UK_spending_by_Dom_All = UK_spending_by_Dom_All.rename(columns={'spend': 'Dom_spend_All'})
    print(UK_spending_by_Dom_All)
else:
    print("DataFrame is empty or missing required columns.")


In [ ]:

# Save the result to a CSV file
csv_filename = "UK_spending_by_Dom_All.csv"
UK_spending_by_Dom_All.to_csv(csv_filename, index=False)

print(f"CSV file '{csv_filename}' has been created successfully.")


In [ ]:
# Summarise the data by UK Cardholder Domestic Online Spending All

UK_spending_by_Dom_Online_All = '''SELECT time_period_value, destination_country, spend 
FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` 
where time_period = 'Quarter' 
and mcg = 'All' 
and merchant_channel = 'Online' 
and cardholder_origin_country = 'All' 
and cardholder_origin = 'UNITED KINGDOM' 
and destination_country = 'UNITED KINGDOM' 
GROUP BY destination_country, 
time_period_value, spend 
ORDER BY time_period_value, destination_country DESC'''
df_by_Dom_Online_All = bq.read_bq_table_sql(client, UK_spending_by_Dom_Online_All)

df_by_Dom_Online_All.head()

# Sum UK Cardholder Domestic Online Spending All Quarterly
# Assuming df_by_Dom_Online_All is the DataFrame returned from the BigQuery query
# Then group by 'time_period_value' and sum the 'spend' for each quarter

# Check if df_by_Dom_Online_All is not None and has the expected columns
if df_by_Dom_Online_All is not None and 'time_period_value' in df_by_Dom_Online_All.columns and 'spend' in df_by_Dom_Online_All.columns:
    # Group by quarter and sum the spend
    UK_spending_by_Dom_Online_All = df_by_Dom_Online_All.groupby('time_period_value')['spend'].sum().reset_index()
   
 # Rename the column
    UK_spending_by_Dom_Online_All = UK_spending_by_Dom_Online_All.rename(columns={'spend': 'Dom_spend_Online_All'})
    print(UK_spending_by_Dom_Online_All)
else:
    print("DataFrame is empty or missing required columns.")

In [ ]:

# Save the result to a CSV file
csv_filename = "UK_spending_by_Dom_Online_All.csv"
UK_spending_by_Dom_Online_All.to_csv(csv_filename, index=False)

print(f"CSV file '{csv_filename}' has been created successfully.")


In [ ]:
# Quaterly Domestic Online Spend Ratio

import pandas as pd

# Load the two CSV files
online_spending = pd.read_csv("UK_spending_by_Dom_Online_All.csv")
total_spending = pd.read_csv("UK_spending_by_Dom_All.csv")

# Merge the two DataFrames on 'time_period_value'
merged_df = pd.merge(online_spending, total_spending, on="time_period_value", how="inner")

# Calculate the online spending ratio
merged_df["online_spending_ratio"] = (merged_df["Dom_spend_Online_All"] / merged_df["Dom_spend_All"]) * 100

# Save the result to a new CSV file
merged_df[["time_period_value", "online_spending_ratio"]].to_csv("Q_Online_Spending_Ratio_Dom.csv", index=False)

print("Online spending ratio by quarter has been saved to 'Q_Online_Spending_Ratio_Dom.csv'.")

In [ ]:
# Summarise the data by country - Abroad Total
# Summarise the data by UK Cardholder Abroad Spending All

UK_spending_by_Intl_All = '''SELECT time_period_value, destination_country, spend 
FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` 
where time_period = 'Quarter' 
and mcc = 'All' 
and mcg = 'All' 
and merchant_channel = 'All' 
and cardholder_origin_country = 'All' 
and cardholder_origin = 'UNITED KINGDOM' 
and destination_country != 'UNITED KINGDOM' 
GROUP BY destination_country, 
time_period_value, spend 
ORDER BY time_period_value, destination_country DESC'''
df_by_Intl_All = bq.read_bq_table_sql(client, UK_spending_by_Intl_All)
df_by_Intl_All = df_by_Intl_All.rename(columns={'spend': 'abroad_spend_all'})
df_by_Intl_All.head()

# UK Cardholder Household Spending Quarterly Abroad Totals

# Assuming df_by_Intl_All is the DataFrame returned from the BigQuery query
# Then group by 'time_period_value' and sum the 'abroad_spend_all' for each quarter

# Check if df_by_Intl_All is not None and has the expected columns
if df_by_Intl_All is not None and 'time_period_value' in df_by_Intl_All.columns and 'abroad_spend_all' in df_by_Intl_All.columns:
    # Group by quarter and sum the total_spend
    Q_spending_by_Intl_All = df_by_Intl_All.groupby('time_period_value')['abroad_spend_all'].sum().reset_index()
   
    print(Q_spending_by_Intl_All)
else:
    print("DataFrame is empty or missing required columns.")
    
    
import pandas as pd

# Check if df_by_Intl_All is defined and has the expected columns
if 'df_by_Intl_All' in globals() and df_by_Intl_All is not None and \
   'time_period_value' in df_by_Intl_All.columns and 'abroad_spend_all' in df_by_Intl_All.columns:
    
    # Group by quarter and sum the abroad_spend_all
    Q_spending_by_Intl_All = df_by_Intl_All.groupby('time_period_value')['abroad_spend_all'].sum().reset_index()
    
    # Save to CSV
    Q_spending_by_Intl_All.to_csv("Q_spending_by_Intl_All.csv", index=False)
    print("CSV file 'Q_spending_by_Intl_All.csv' has been created successfully.")
else:
    print("DataFrame is empty or missing required columns.")

In [ ]:
# Summarise the data by country - Quarterly Abroad Online
# Summarise the data by UK Cardholder Abroad Online Spending All 

UK_spending_by_online_Intl_All = '''SELECT time_period_value, destination_country, spend 
FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` 
where time_period = 'Quarter' 
and mcg = 'All' 
and merchant_channel = 'Online' 
and cardholder_origin_country = 'All' 
and cardholder_origin = 'UNITED KINGDOM' 
and destination_country != 'UNITED KINGDOM' 
GROUP BY destination_country, 
time_period_value, spend 
ORDER BY time_period_value, destination_country DESC'''
df_by_online_Intl_All = bq.read_bq_table_sql(client, UK_spending_by_online_Intl_All)
df_by_online_Intl_All = df_by_online_Intl_All.rename(columns={'spend': 'online_Intl_All'})
df_by_online_Intl_All.head()

# UK Cardholder Household Online Spending Quarterly Abroad Totals

# Assuming df_by_online_Intl_All is the DataFrame returned from the BigQuery query
# Then group by 'time_period_value' and sum the 'total_spend' for each quarter

# Check if df_by_online_Intl_All is not None and has the expected columns
if df_by_online_Intl_All is not None and 'time_period_value' in df_by_online_Intl_All.columns and 'online_Intl_All' in df_by_online_Intl_All.columns:
    # Group by quarter and sum the total_spend
    UK_spending_by_online_Intl_All = df_by_online_Intl_All.groupby('time_period_value')['online_Intl_All'].sum().reset_index()
   
    print(UK_spending_by_online_Intl_All)
else:
    print("DataFrame is empty or missing required columns.")
    
import pandas as pd

# Check if df_by_online_Intl_All is defined and has the expected columns
if 'df_by_online_Intl_All' in globals() and df_by_online_Intl_All is not None and \
   'time_period_value' in df_by_online_Intl_All.columns and 'online_Intl_All' in df_by_online_Intl_All.columns:
    
    # Group by quarter and sum the abroad_spend_all
    Q_spending_by_online_Intl_All = df_by_online_Intl_All.groupby('time_period_value')['online_Intl_All'].sum().reset_index()
    
    # Save to CSV
    Q_spending_by_online_Intl_All.to_csv("Q_spending_by_Online_Intl_All.csv", index=False)
    print("CSV file 'Q_spending_by_Online_Intl_All.csv' has been created successfully.")
else:
    print("DataFrame is empty or missing required columns.") 

In [ ]:
# Quaterly Abroad Online Spend Ratio

import pandas as pd

# Load the two CSV files
online_spending = pd.read_csv("Q_spending_by_Online_Intl_All.csv")
total_spending = pd.read_csv("Q_spending_by_Intl_All.csv")

# Merge the two DataFrames on 'time_period_value'
merged_df = pd.merge(online_spending, total_spending, on="time_period_value", how="inner")

# Calculate the online spending ratio
merged_df["online_spending_ratio"] = (merged_df["online_Intl_All"] / merged_df["abroad_spend_all"]) * 100

# Save the result to a new CSV file
merged_df[["time_period_value", "online_spending_ratio"]].to_csv("Q_Online_Spending_Ratio_Intl.csv", index=False)

print("Online spending ratio by quarter has been saved to 'Q_Online_Spending_Ratio_Intl.csv'.")


In [ ]:
# Quarterly Comparison for UK Cardholder Domestic vs Abroad Online Spending Ratios

import pandas as pd
import matplotlib.pyplot as plt

# Load the CSV files
intl_ratio_file = "Q_Online_Spending_Ratio_Intl.csv"
dom_ratio_file = "Q_Online_Spending_Ratio_Dom.csv"

# Read the data
intl_df = pd.read_csv(intl_ratio_file)
dom_df = pd.read_csv(dom_ratio_file)

# Ensure the time_period_value column is sorted and consistent
intl_df = intl_df.sort_values("time_period_value")
dom_df = dom_df.sort_values("time_period_value")

# Merge the two datasets on time_period_value
merged_df = pd.merge(intl_df, dom_df, on="time_period_value", how="inner")

# Plot the indexed trends
plt.figure(figsize=(12, 6))
plt.plot(merged_df["time_period_value"], merged_df.iloc[:, 1], label="International Online Spending Ratio", marker='o')
plt.plot(merged_df["time_period_value"], merged_df.iloc[:, 2], label="Domestic Online Spending Ratio", marker='o')
plt.xticks(rotation=45)
plt.xlabel("Quarter")
plt.ylabel("Online Spending Ratio (%)")
plt.title("Quarterly Online Spending Ratio: International vs Domestic")
plt.legend()
plt.tight_layout()
plt.grid(True)
plt.savefig("Online_Spending_Ratio_Comparison.png")
plt.show()

